# PyTorch Hands-On Tutorial for Computational Materials
## From Tensor Algebra to Neural Networks and Scientific Machine Learning

**Level:** IIT M.Tech / PhD Applied Materials / Computational Materials  
**Prerequisites:** Python fundamentals, NumPy, linear algebra, probability/statistics, basic machine learning  
**Recommended position:** After the Python, NumPy, data-analysis, mathematics, numerical-methods, and classical-ML modules

### Course philosophy

This tutorial does not treat PyTorch as a collection of commands.

Students should understand the computational ideas behind:

**tensor → operation → computational graph → gradient → optimization → model → scientific prediction**

The examples are deliberately connected to materials science, numerical simulation, microstructure analysis, and materials-property prediction.


# Learning objectives

By the end of this tutorial, students should be able to:

1. Explain what a PyTorch tensor is.
2. Create, reshape, index, and manipulate tensors.
3. Understand tensor dtypes and devices.
4. Perform vectorized mathematical operations.
5. Use matrix multiplication and broadcasting.
6. Understand PyTorch's automatic differentiation.
7. Construct and inspect computational graphs conceptually.
8. Implement gradient descent from first principles.
9. Build neural networks with `torch.nn`.
10. Train regression and classification models.
11. Build custom datasets and DataLoaders.
12. Use validation and test datasets correctly.
13. Save and load trained models.
14. Use GPU acceleration when available.
15. Understand how PyTorch supports scientific machine learning and differentiable numerical models.


# 1. Environment and imports

The notebook can be used with Jupyter, JupyterLab, or Google Colab.

If PyTorch is not installed in your environment, install the appropriate PyTorch build before proceeding. In managed environments such as Colab, PyTorch is commonly preinstalled.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 2. What is a tensor?

A tensor is a multidimensional numerical array.

Examples:

- scalar → 0D tensor
- vector → 1D tensor
- matrix → 2D tensor
- image → typically 3D tensor `(channels, height, width)`
- batch of images → 4D tensor
- atomistic data → potentially higher-dimensional tensors

This is closely connected to NumPy arrays.


In [ ]:
scalar = torch.tensor(5.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

print("scalar:", scalar)
print("vector:", vector)
print("matrix:")
print(matrix)

print("Shapes:")
print(scalar.shape)
print(vector.shape)
print(matrix.shape)


# 3. Tensor attributes

Important attributes include:

- `.shape`
- `.ndim`
- `.dtype`
- `.device`

These become essential when training neural networks.


In [ ]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])

print("shape:", x.shape)
print("ndim:", x.ndim)
print("dtype:", x.dtype)
print("device:", x.device)


# 4. Creating tensors

Common methods:

- `torch.tensor`
- `torch.zeros`
- `torch.ones`
- `torch.full`
- `torch.arange`
- `torch.linspace`
- `torch.rand`
- `torch.randn`


In [ ]:
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.full((2, 2), 7.0))
print(torch.arange(0, 10, 2))
print(torch.linspace(0, 1, 6))
print(torch.rand(2, 3))
print(torch.randn(2, 3))


# Exercise 1 — Tensor construction

Create:

1. a vector containing temperatures from 300 K to 1200 K in steps of 100 K,
2. a `4 × 4` identity matrix,
3. a `3 × 5` matrix containing random normally distributed values,
4. a tensor containing the values `[0.1, 0.2, ..., 1.0]`.


# 5. Converting between NumPy and PyTorch


In [ ]:
array = np.array([
    [1.0, 2.0],
    [3.0, 4.0]
])

tensor = torch.from_numpy(array)

print("NumPy:")
print(array)

print("PyTorch:")
print(tensor)

back_to_numpy = tensor.numpy()
print("Back to NumPy:")
print(back_to_numpy)


# 6. Tensor dtypes

Neural networks usually use floating-point tensors.

Common choices include:

- `torch.float32`
- `torch.float64`

Integer tensors are useful for indices and class labels.


In [ ]:
x32 = torch.tensor([1, 2, 3], dtype=torch.float32)
x64 = torch.tensor([1, 2, 3], dtype=torch.float64)
xint = torch.tensor([1, 2, 3], dtype=torch.int64)

print(x32.dtype)
print(x64.dtype)
print(xint.dtype)


### Scientific-computing note

`float64` can be important when numerical precision matters.

Deep-learning workloads often use `float32` because it is faster and uses less memory.

For scientific machine learning, the precision choice should be deliberate.


# 7. Indexing and slicing

PyTorch indexing is similar to NumPy.


In [ ]:
x = torch.arange(1, 13).reshape(3, 4)

print(x)

print("First row:", x[0])
print("Second column:", x[:, 1])
print("Top-left block:")
print(x[:2, :2])
print("Last row:", x[-1])


# 8. Reshaping tensors

Common operations:

- `.reshape()`
- `.view()`
- `.flatten()`
- `.squeeze()`
- `.unsqueeze()`

These are extremely important when moving data between scientific representations and neural-network layers.


In [ ]:
x = torch.arange(12)

print("Original:", x.shape)

a = x.reshape(3, 4)
print("3 x 4:", a.shape)

b = x.reshape(2, 2, 3)
print("2 x 2 x 3:", b.shape)

print("Flattened:", b.flatten().shape)

print("Unsqueezed:", x.unsqueeze(0).shape)


# Exercise 2 — Reshape a simulation field

Suppose a 1D temperature field contains 100 values.

Create a tensor of 100 values and reshape it into:

- `(10, 10)`
- `(2, 5, 10)`
- `(1, 100)`

Explain which representation would be appropriate for a 2D field and which would be appropriate for a batch of one 1D sample.


# 9. Element-wise operations

PyTorch supports vectorized operations similar to NumPy.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

print("x^2:", x**2)
print("sqrt(x):", torch.sqrt(x))
print("exp(x):", torch.exp(x))
print("sin(x):", torch.sin(x))
print("log(x):", torch.log(x))


# 10. Matrix multiplication

Use `@` or `torch.matmul()`.

Matrix multiplication is fundamental to neural networks:

`z = Wx + b`


In [ ]:
W = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

x = torch.tensor([5.0, 6.0])

print(W @ x)
print(torch.matmul(W, x))


# 11. Broadcasting

Broadcasting allows operations between tensors of compatible shapes.

This is extensively used for:

- normalization,
- batch processing,
- adding biases,
- coordinate transformations,
- physical parameter fields.


In [ ]:
X = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0]
])

offset = torch.tensor([10.0, 20.0, 30.0])

print(X + offset)


# Exercise 3 — Feature normalization

Given a tensor containing 5 samples and 3 features, calculate the mean and standard deviation of each feature and standardize the data using tensor operations.

Do not use scikit-learn for this exercise.


# 12. Tensor reductions

Important operations:

- `.sum()`
- `.mean()`
- `.std()`
- `.min()`
- `.max()`
- `.argmin()`
- `.argmax()`


In [ ]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

print("Mean:", x.mean())
print("Mean by column:", x.mean(dim=0))
print("Mean by row:", x.mean(dim=1))
print("Column sum:", x.sum(dim=0))
print("Maximum:", x.max())


# 13. Device management

PyTorch can execute operations on CPUs or GPUs.

A common pattern is:

`tensor.to(device)`

and

`model.to(device)`.


In [ ]:
x = torch.randn(1000, 1000).to(device)

print(x.device)
print("Mean:", x.mean().item())


### Important rule

Tensors involved in the same operation generally need to be on compatible devices.

For example, a CPU tensor cannot directly be matrix-multiplied with a CUDA tensor.


# 14. Performance experiment — CPU versus GPU

Run this cell if a GPU is available.

The exact speedup depends strongly on hardware, tensor size, memory transfer, and workload.


In [ ]:
import time

size = 3000

a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.perf_counter()
c_cpu = a_cpu @ b_cpu
cpu_time = time.perf_counter() - start

print(f"CPU time: {cpu_time:.3f} s")

if torch.cuda.is_available():
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    torch.cuda.synchronize()
    start = time.perf_counter()

    c_gpu = a_gpu @ b_gpu

    torch.cuda.synchronize()
    gpu_time = time.perf_counter() - start

    print(f"GPU time: {gpu_time:.3f} s")
    print(f"Approximate speedup: {cpu_time / gpu_time:.2f}x")
else:
    print("GPU not available.")


# 15. Automatic differentiation

One of PyTorch's most important capabilities is automatic differentiation.

Suppose:

`y = x^3 + 2x^2`

Then:

`dy/dx = 3x^2 + 4x`.

PyTorch can calculate the derivative automatically.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x**3 + 2*x**2

y.backward()

print("y =", y.item())
print("dy/dx =", x.grad.item())


# 16. Understanding `requires_grad`

When `requires_grad=True`, PyTorch tracks operations involving the tensor so that gradients can later be calculated.

This is the foundation of:

- neural-network training,
- optimization,
- inverse problems,
- differentiable physics,
- physics-informed neural networks.


# 17. A multivariable example

Let

`f(x,y) = x² + 3xy + y²`.

Then:

`∂f/∂x = 2x + 3y`

`∂f/∂y = 3x + 2y`.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x**2 + 3*x*y + y**2

f.backward()

print("df/dx =", x.grad.item())
print("df/dy =", y.grad.item())


# Exercise 4 — Differentiate a physical model

Consider:

`E(x) = 0.5*k*x²`

where `E` is elastic energy and `x` is displacement.

Use PyTorch to calculate:

`dE/dx`

for `k = 1000 N/m` and `x = 0.02 m`.

Then verify the result analytically.


# 18. Gradients and optimization

Consider minimizing:

`f(x) = (x - 4)^2`.

The minimum is at `x = 4`.

We can use gradient descent:

`x_new = x_old - η df/dx`.


In [ ]:
x = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

history = []

for step in range(50):
    f = (x - 4)**2

    f.backward()

    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_()

    history.append((x.item(), f.item()))

print("Final x:", x.item())
print("Final f:", f.item())


# 19. Visualize gradient descent


In [ ]:
history = np.array(history)

plt.figure(figsize=(8, 5))
plt.plot(history[:, 0], marker="o", markersize=3)
plt.axhline(4, linestyle="--")
plt.xlabel("Iteration")
plt.ylabel("x")
plt.title("Gradient descent")
plt.grid(alpha=0.25)
plt.show()


# 20. Why `torch.no_grad()`?

Parameter updates should not themselves become part of the differentiation graph.

Therefore updates are commonly performed inside:

```python
with torch.no_grad():
    ...
```

Later, PyTorch optimizers such as Adam perform this update mechanism for us.


# 21. The neural-network connection

A linear layer computes:

`y = xW^T + b`.

PyTorch implements this through `nn.Linear`.

A neural network is essentially a composition of tensor operations whose parameters are optimized using gradients.


In [ ]:
layer = nn.Linear(3, 2)

print(layer)
print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)


# 22. Forward pass through a linear layer


In [ ]:
layer = nn.Linear(3, 2)

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

y = layer(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)
print(y)


# 23. Building an MLP

A multilayer perceptron can be constructed using `nn.Sequential`.

Example:

`input → Linear → ReLU → Linear → ReLU → Linear → output`


In [ ]:
model = nn.Sequential(
    nn.Linear(5, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

print(model)


# 24. Inspecting model parameters


In [ ]:
total_parameters = 0

for name, parameter in model.named_parameters():
    print(name, parameter.shape, parameter.numel())
    total_parameters += parameter.numel()

print("Total parameters:", total_parameters)


# 25. Synthetic materials-property dataset

We create a synthetic dataset representing descriptors that might appear in a materials-informatics problem:

- atomic-size mismatch
- electronegativity
- density
- grain size
- processing temperature
- cooling rate
- defect fraction
- elastic modulus

The target is a nonlinear synthetic property.

The purpose is to learn the PyTorch workflow, not to claim that the synthetic equation is a physical law.


In [ ]:
rng = np.random.default_rng(42)
n = 4000

df = pd.DataFrame({
    "size_mismatch": rng.uniform(0.0, 0.18, n),
    "electronegativity": rng.normal(1.8, 0.3, n),
    "density": rng.normal(7.0, 0.8, n),
    "grain_size": rng.lognormal(np.log(12), 0.5, n),
    "temperature": rng.normal(1100, 150, n),
    "cooling_rate": rng.lognormal(np.log(20), 0.8, n),
    "defect_fraction": np.clip(
        rng.lognormal(np.log(0.01), 0.7, n),
        0, 0.15
    ),
    "modulus": rng.normal(180, 30, n)
})

df["property"] = (
    120
    + 0.8 * df["modulus"]
    + 100 / np.sqrt(df["grain_size"])
    - 250 * df["size_mismatch"]**2
    + 60 * np.sin(df["electronegativity"])
    + 0.07 * df["temperature"]
    - 20 * np.log1p(df["cooling_rate"])
    - 1000 * df["defect_fraction"]
    + 150 * df["size_mismatch"]
      * np.sin(df["temperature"] / 180)
    + rng.normal(0, 25, n)
)

df.head()


# 26. Train / validation / test split

A scientifically meaningful split is essential.

Use:

- training data → fit model parameters
- validation data → select hyperparameters
- test data → final evaluation

The test set should not influence model design.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["property"]).to_numpy(dtype=np.float32)
y = df["property"].to_numpy(dtype=np.float32).reshape(-1, 1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# 27. Feature scaling

Neural networks usually train more reliably when continuous features have comparable scales.

Standardization:

`z = (x - μ) / σ`

The scaler must be fitted using **training data only**.


In [ ]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)


# 28. Convert data to tensors


In [ ]:
Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32)

Xv = torch.tensor(X_val_s, dtype=torch.float32)
yv = torch.tensor(y_val, dtype=torch.float32)

Xte = torch.tensor(X_test_s, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.float32)

print(Xtr.shape, ytr.shape)


# 29. DataLoader

A DataLoader handles:

- mini-batches,
- shuffling,
- iteration.

Mini-batch training is central to practical deep learning.


In [ ]:
train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(Xv, yv),
    batch_size=256
)

test_loader = DataLoader(
    TensorDataset(Xte, yte),
    batch_size=256
)

xb, yb = next(iter(train_loader))

print("Batch X:", xb.shape)
print("Batch y:", yb.shape)


# 30. Define a materials-property model


In [ ]:
class MaterialsMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

model = MaterialsMLP(Xtr.shape[1]).to(device)

print(model)


# 31. Loss function

For regression, use mean squared error:

`MSE = mean((y - y_hat)^2)`.

PyTorch:

`nn.MSELoss()`.


In [ ]:
loss_fn = nn.MSELoss()

example_prediction = model(Xtr[:10].to(device))
example_loss = loss_fn(example_prediction, ytr[:10].to(device))

print("Example loss:", example_loss.item())


# 32. Optimizer

The optimizer updates trainable parameters.

Common optimizers:

- SGD
- Adam
- AdamW

Start with Adam for practical experiments, then compare it with SGD to understand the differences.


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print(optimizer)


# 33. The PyTorch training loop

The fundamental sequence is:

1. get a batch,
2. move it to the device,
3. zero old gradients,
4. forward pass,
5. calculate loss,
6. backpropagate,
7. update parameters.


In [ ]:
epochs = 150
train_losses = []

for epoch in range(epochs):

    model.train()

    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        prediction = model(xb)

        loss = loss_fn(prediction, yb)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item() * len(xb)

    epoch_loss /= len(train_loader.dataset)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 25 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Training MSE = {epoch_loss:.3f}"
        )


# 34. Learning curve


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("PyTorch training curve")
plt.grid(alpha=0.25)
plt.show()


# 35. Validation loop

Validation is performed without changing model parameters.

Use:

- `model.eval()`
- `torch.no_grad()`


In [ ]:
model.eval()

validation_losses = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        prediction = model(xb)
        loss = loss_fn(prediction, yb)

        validation_losses.append(
            loss.item() * len(xb)
        )

val_loss = sum(validation_losses) / len(val_loader.dataset)

print("Validation MSE:", val_loss)


# 36. Regression metrics


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()

predictions = []

with torch.no_grad():
    for xb, yb in test_loader:
        predictions.append(
            model(xb.to(device)).cpu().numpy()
        )

predictions = np.vstack(predictions).ravel()
targets = y_test.ravel()

mae = mean_absolute_error(targets, predictions)
rmse = np.sqrt(mean_squared_error(targets, predictions))
r2 = r2_score(targets, predictions)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


# 37. Predicted versus true


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(targets, predictions, alpha=0.5)

lo = min(targets.min(), predictions.min())
hi = max(targets.max(), predictions.max())

plt.plot([lo, hi], [lo, hi], "--")

plt.xlabel("True property")
plt.ylabel("Predicted property")
plt.title("PyTorch materials-property regression")
plt.grid(alpha=0.25)
plt.show()


# Exercise 5 — Change the architecture

Create and compare at least three networks:

### Model A
`8 → 16 → 1`

### Model B
`8 → 64 → 32 → 1`

### Model C
`8 → 128 → 64 → 32 → 1`

Compare:

- parameter count,
- training time,
- validation error,
- test error.

Discuss whether the largest network is necessarily the best.


# 38. Custom Dataset class

For simple tabular data, `TensorDataset` is sufficient.

For complex scientific data, students often need a custom `Dataset`.

Typical examples:

- microscopy images,
- simulation snapshots,
- atomistic structures,
- variable-length structures,
- paired input/output fields.


In [ ]:
class MaterialsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(
            X, dtype=torch.float32
        )
        self.y = torch.tensor(
            y, dtype=torch.float32
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


custom_dataset = MaterialsDataset(
    X_train_s, y_train
)

print("Dataset length:", len(custom_dataset))

sample_x, sample_y = custom_dataset[0]

print("Sample X:", sample_x)
print("Sample y:", sample_y)


# 39. Saving and loading models

For reproducible scientific work, save:

- model state,
- architecture information,
- feature-scaling information,
- training configuration,
- random seeds,
- dataset/version information.

PyTorch commonly saves model parameters with `state_dict()`.


In [ ]:
model_path = "materials_mlp_state.pt"

torch.save(
    model.state_dict(),
    model_path
)

print("Saved:", model_path)


In [ ]:
loaded_model = MaterialsMLP(
    Xtr.shape[1]
).to(device)

loaded_model.load_state_dict(
    torch.load(model_path, map_location=device)
)

loaded_model.eval()

with torch.no_grad():
    test_output = loaded_model(
        Xte[:5].to(device)
    )

print(test_output.cpu())


# 40. Reproducibility

Deep-learning experiments can change because of:

- random initialization,
- shuffled mini-batches,
- train/validation splits,
- hardware,
- library versions.

Use explicit random seeds for teaching and controlled experiments.

For serious computational studies, document the environment and software versions as well.


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

print("Seeds reset.")


# 41. Classification with PyTorch

For binary classification, the network produces a logit.

Use:

`nn.BCEWithLogitsLoss()`

rather than manually applying a sigmoid before the loss.


In [ ]:
median_property = df["property"].median()

classification_target = (
    df["property"] > median_property
).astype(np.float32).to_numpy().reshape(-1, 1)

Xc = df.drop(columns=["property"]).to_numpy(
    dtype=np.float32
)

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc,
    classification_target,
    test_size=0.20,
    random_state=42,
    stratify=classification_target
)

classifier_scaler = StandardScaler()

Xc_train = classifier_scaler.fit_transform(Xc_train)
Xc_test = classifier_scaler.transform(Xc_test)

Xc_train = torch.tensor(Xc_train, dtype=torch.float32)
Xc_test = torch.tensor(Xc_test, dtype=torch.float32)

yc_train = torch.tensor(yc_train, dtype=torch.float32)
yc_test = torch.tensor(yc_test, dtype=torch.float32)

classification_loader = DataLoader(
    TensorDataset(Xc_train, yc_train),
    batch_size=64,
    shuffle=True
)


In [ ]:
class MaterialsClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

classifier = MaterialsClassifier(
    Xc_train.shape[1]
).to(device)

classification_loss = nn.BCEWithLogitsLoss()

classification_optimizer = torch.optim.Adam(
    classifier.parameters(),
    lr=1e-3
)

for epoch in range(100):
    classifier.train()

    for xb, yb in classification_loader:
        xb, yb = xb.to(device), yb.to(device)

        classification_optimizer.zero_grad()

        logits = classifier(xb)
        loss = classification_loss(logits, yb)

        loss.backward()
        classification_optimizer.step()

classifier.eval()

with torch.no_grad():
    logits = classifier(
        Xc_test.to(device)
    ).cpu().numpy().ravel()

probabilities = 1 / (1 + np.exp(-logits))
predicted_classes = (probabilities >= 0.5).astype(int)

accuracy = (
    predicted_classes
    == yc_test.numpy().ravel()
).mean()

print("Accuracy:", accuracy)


# 42. Dropout and regularization

Dropout randomly removes activations during training.

It can help reduce overfitting.

However:

- it does not fix data leakage,
- it does not replace validation,
- it does not guarantee physical generalization.


In [ ]:
class RegularizedMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

regularized_model = RegularizedMLP(
    Xtr.shape[1]
).to(device)

print(regularized_model)


# 43. Training versus evaluation mode

Some layers behave differently during training and evaluation.

Important calls:

`model.train()`

`model.eval()`

Dropout is the clearest example.

Always use `model.eval()` during validation and testing.


# 44. Learning-rate scheduling

The learning rate can be reduced during training.

Example strategy:

`ReduceLROnPlateau`

This can be useful when validation performance stops improving.


In [ ]:
scheduler_model = MaterialsMLP(
    Xtr.shape[1]
).to(device)

scheduler_optimizer = torch.optim.Adam(
    scheduler_model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    scheduler_optimizer,
    mode="min",
    factor=0.5,
    patience=10
)

print(scheduler)


# 45. Autograd for a physics equation

Automatic differentiation is especially powerful in scientific machine learning.

Consider the 1D temperature field:

`T(x) = sin(πx)`.

The second derivative is:

`d²T/dx² = -π² sin(πx)`.

PyTorch can calculate this derivative automatically.


In [ ]:
x = torch.linspace(
    0, 1, 100,
    dtype=torch.float32,
    requires_grad=True
)

T = torch.sin(torch.pi * x)

dT_dx = torch.autograd.grad(
    T,
    x,
    grad_outputs=torch.ones_like(T),
    create_graph=True
)[0]

d2T_dx2 = torch.autograd.grad(
    dT_dx,
    x,
    grad_outputs=torch.ones_like(dT_dx),
    create_graph=True
)[0]

analytic = -torch.pi**2 * torch.sin(torch.pi * x)

error = torch.max(
    torch.abs(d2T_dx2 - analytic)
).item()

print("Maximum derivative error:", error)


# 46. Why this matters: PDEs

For a PDE such as the heat equation,

`∂T/∂t = α ∂²T/∂x²`,

automatic differentiation can calculate spatial and temporal derivatives of a neural-network approximation `T(x,t)`.

This is one foundation of Physics-Informed Neural Networks (PINNs).

The network can be trained not only against measured data, but also against a physical residual such as:

`R = ∂T/∂t - α ∂²T/∂x²`.


# 47. Mini-example — PDE residual with a neural network

We create a small neural network representing `T(x,t)`.

The goal here is only to calculate derivatives and a PDE residual. It is **not yet a complete PINN solver**.


In [ ]:
class TemperatureNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)


pde_model = TemperatureNet().to(device)

n_points = 200

x = torch.rand(n_points, 1, device=device, requires_grad=True)
t = torch.rand(n_points, 1, device=device, requires_grad=True)

T = pde_model(x, t)

dT_dx, dT_dt = torch.autograd.grad(
    T,
    (x, t),
    grad_outputs=torch.ones_like(T),
    create_graph=True
)

d2T_dx2 = torch.autograd.grad(
    dT_dx,
    x,
    grad_outputs=torch.ones_like(dT_dx),
    create_graph=True
)[0]

alpha = 0.01

residual = dT_dt - alpha * d2T_dx2

print("PDE residual shape:", residual.shape)
print("Initial residual MSE:",
      torch.mean(residual**2).item())


# Exercise 6 — Heat-equation residual

Modify the previous cell to verify the PDE residual for the analytical solution

`T(x,t) = exp(-π² α t) sin(πx)`.

Calculate:

- `T`
- `∂T/∂t`
- `∂²T/∂x²`
- `R = ∂T/∂t - α∂²T/∂x²`

Show that `R` is approximately zero.


# 48. CNN introduction

Images and microstructures can be represented as tensors.

A grayscale image batch has shape:

`(batch, channels, height, width)`.

Example:

`(32, 1, 128, 128)`

means:

- 32 images
- 1 channel
- 128 × 128 pixels


In [ ]:
images = torch.randn(16, 1, 32, 32)

conv = nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3,
    padding=1
)

features = conv(images)

print("Input:", images.shape)
print("Output:", features.shape)


# 49. Synthetic microstructure example

Create simple binary microstructures and use a CNN to predict a synthetic property.

This introduces the connection:

**microstructure image → convolutional features → material property**


In [ ]:
def make_microstructure(size=32, n_circles=8):
    image = np.zeros((size, size), dtype=np.float32)

    yy, xx = np.mgrid[:size, :size]

    for _ in range(n_circles):
        cx = rng.integers(0, size)
        cy = rng.integers(0, size)
        radius = rng.integers(2, 6)

        image[
            (xx-cx)**2 + (yy-cy)**2 <= radius**2
        ] = 1.0

    return image

images = np.array([
    make_microstructure()
    for _ in range(1200)
])

target = (
    200
    + 500 * images.mean(axis=(1, 2))
    + rng.normal(0, 10, len(images))
)

print(images.shape)


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, image in zip(axes, images[:5]):
    ax.imshow(image)
    ax.axis("off")

plt.suptitle("Synthetic microstructures")
plt.show()


# 50. CNN architecture

A simple CNN:

`image → convolution → ReLU → pooling → convolution → ReLU → pooling → fully connected → property`


In [ ]:
class MicrostructureCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x)

cnn = MicrostructureCNN().to(device)

print(cnn)


# 51. Data preparation for CNN


In [ ]:
from sklearn.model_selection import train_test_split

img_train, img_test, y_img_train, y_img_test = train_test_split(
    images,
    target,
    test_size=0.20,
    random_state=42
)

img_train = torch.tensor(
    img_train[:, None, :, :],
    dtype=torch.float32
)

img_test = torch.tensor(
    img_test[:, None, :, :],
    dtype=torch.float32
)

y_img_train = torch.tensor(
    y_img_train.reshape(-1, 1),
    dtype=torch.float32
)

y_img_test = torch.tensor(
    y_img_test.reshape(-1, 1),
    dtype=torch.float32
)

img_loader = DataLoader(
    TensorDataset(img_train, y_img_train),
    batch_size=32,
    shuffle=True
)


# 52. Train the CNN


In [ ]:
cnn_loss = nn.MSELoss()

cnn_optimizer = torch.optim.Adam(
    cnn.parameters(),
    lr=1e-3
)

cnn_history = []

for epoch in range(60):
    cnn.train()
    total_loss = 0.0

    for xb, yb in img_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        cnn_optimizer.zero_grad()

        prediction = cnn(xb)
        loss = cnn_loss(prediction, yb)

        loss.backward()
        cnn_optimizer.step()

        total_loss += loss.item() * len(xb)

    epoch_loss = total_loss / len(img_loader.dataset)
    cnn_history.append(epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d}: "
            f"MSE = {epoch_loss:.3f}"
        )


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cnn_history)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("CNN training")
plt.grid(alpha=0.25)
plt.show()


# 53. CNN evaluation


In [ ]:
cnn.eval()

with torch.no_grad():
    cnn_pred = cnn(
        img_test.to(device)
    ).cpu().numpy().ravel()

cnn_true = y_img_test.numpy().ravel()

print(
    "CNN RMSE:",
    np.sqrt(mean_squared_error(cnn_true, cnn_pred))
)

print(
    "CNN R²:",
    r2_score(cnn_true, cnn_pred)
)


# 54. Comparing physics-based computation and neural networks

PyTorch is not a replacement for numerical methods.

A strong computational-materials workflow may be:

1. derive a physical model,
2. implement a numerical solver,
3. generate simulation data,
4. train a surrogate neural network,
5. validate against the original solver,
6. investigate speedup and error,
7. enforce physical constraints where appropriate.

This distinction is essential in scientific machine learning.


# 55. Suggested PyTorch ecosystem

After mastering this notebook, students can explore:

### Core
- PyTorch
- TorchVision
- TorchMetrics

### Scientific ML
- automatic differentiation
- PINNs
- neural operators
- differentiable programming

### Graph learning
- PyTorch Geometric
- DGL

### Materials / atomistic ML
- pymatgen
- ASE
- matminer
- matgl
- atomistic graph/equivariant frameworks such as MACE-family tools

The mathematical concepts learned here remain useful regardless of framework.


# 56. Common PyTorch errors

### Error 1: Shape mismatch

Example:

`mat1 and mat2 shapes cannot be multiplied`

Check the dimensions of the tensors entering the layer.

### Error 2: CPU/GPU mismatch

Move both tensors/model to the same device.

### Error 3: Wrong dtype

Neural-network inputs are commonly `float32`.

### Error 4: Forgetting `zero_grad()`

Gradients accumulate by default.

### Error 5: Forgetting `model.eval()`

Important for dropout and batch normalization.

### Error 6: Applying sigmoid before `BCEWithLogitsLoss`

Do not manually apply sigmoid when using `BCEWithLogitsLoss`.


# 57. Debugging checklist

When a PyTorch model behaves unexpectedly:

1. Print tensor shapes.
2. Print tensor dtypes.
3. Check for NaN/Inf values.
4. Check feature scaling.
5. Check target scaling.
6. Check the loss definition.
7. Check learning rate.
8. Check whether gradients are nonzero.
9. Check train/evaluation mode.
10. Check train/test leakage.
11. Test on a tiny dataset that the model can overfit.
12. Compare against a simple baseline.


# 58. Exercise 7 — Overfit a tiny dataset

Take only 20 samples from the materials-property dataset.

Build an MLP and train it until the training error becomes extremely small.

### Questions

1. Can the network memorize the data?
2. What happens to validation performance?
3. Why does this demonstrate model capacity rather than generalization?


# 59. Exercise 8 — Compare optimizers

Train the same MLP with:

- SGD
- SGD with momentum
- Adam
- AdamW

Keep the architecture and data split fixed.

Compare:

- convergence speed,
- final validation error,
- stability of the learning curve.

Do not change several hyperparameters simultaneously.


# 60. Exercise 9 — Gradient inspection

After one training step, inspect:

`parameter.grad`

for every trainable parameter.

Questions:

1. Which parameters receive gradients?
2. What happens if an activation is removed?
3. Why must the computational graph connect the loss to the parameters?


In [ ]:
small_model = MaterialsMLP(
    Xtr.shape[1]
).to(device)

xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)

optimizer = torch.optim.Adam(
    small_model.parameters(),
    lr=1e-3
)

optimizer.zero_grad()

prediction = small_model(xb)
loss = loss_fn(prediction, yb)

loss.backward()

for name, parameter in small_model.named_parameters():
    if parameter.grad is not None:
        print(
            name,
            "gradient norm =",
            parameter.grad.norm().item()
        )


# 61. Mini-project — Neural materials-property predictor

Build a complete PyTorch regression pipeline.

### Required components

1. dataset description
2. scientific motivation
3. train/validation/test split
4. feature scaling
5. DataLoader
6. baseline model
7. MLP
8. training loop
9. validation loop
10. learning curves
11. test metrics
12. predicted-vs-true plot
13. residual analysis
14. hyperparameter experiment
15. model saving/loading
16. discussion of limitations

### Required comparison

Compare your neural network against at least one classical ML model.


# 62. Mini-project — CNN for microstructure

Use real or synthetic microstructures.

Possible targets:

- phase fraction
- porosity
- grain size
- hardness
- yield strength
- thermal conductivity

Compare:

**handcrafted descriptors + classical ML**

against

**CNN + learned representation**.

Discuss the amount of data required and the scientific meaning of the learned representation.


# 63. Mini-project — PyTorch surrogate for a numerical solver

Use the finite-difference heat/diffusion solver from the numerical-methods module.

Generate data for different:

- diffusivities,
- initial conditions,
- boundary conditions,
- time points.

Train a neural network to approximate the numerical solution.

### Evaluate

- pointwise error,
- field-level RMSE,
- extrapolation,
- inference time,
- physical consistency.

The numerical solver remains the reference model.


# 64. Mini-project — Differentiable materials model

Choose a differentiable mathematical model.

Examples:

- elastic energy,
- diffusion,
- heat transfer,
- phase-field free energy,
- simple constitutive relation.

Use PyTorch autograd to calculate derivatives.

Then formulate an optimization or inverse problem.

Example:

**given measured displacement and force → estimate material parameter `E`.**


# 65. Capstone extension — Materials scientific machine learning

A strong final project can combine the entire course:

**materials physics**
→ **numerical simulation**
→ **data generation**
→ **feature engineering**
→ **classical ML**
→ **PyTorch deep learning**
→ **scientific validation**

Possible final systems:

- neural surrogate for diffusion,
- CNN microstructure-property model,
- atomistic graph model,
- differentiable constitutive model,
- physics-informed heat/diffusion solver,
- materials screening model.


# 66. Assessment rubric

### Mathematical understanding — 25%

- tensors
- matrix operations
- computational graphs
- derivatives
- gradient descent

### PyTorch implementation — 25%

- tensor manipulation
- autograd
- `nn.Module`
- DataLoader
- training loop

### ML methodology — 20%

- data splitting
- scaling
- validation
- regularization
- metrics

### Scientific reasoning — 20%

- physical assumptions
- data leakage
- domain of applicability
- comparison with baselines
- limitations

### Reproducibility — 10%

- random seeds
- saved model
- documented environment
- clear notebook structure


# 67. Viva questions

Be prepared to answer:

1. What is a tensor?
2. Why does PyTorch use tensors instead of ordinary Python lists?
3. What is broadcasting?
4. What is matrix multiplication in a neural network?
5. What does `requires_grad=True` mean?
6. What is a computational graph?
7. What does `.backward()` calculate?
8. Why do we call `optimizer.zero_grad()`?
9. What does `optimizer.step()` do?
10. Why do we use `model.train()` and `model.eval()`?
11. Why should feature scaling be fitted only on training data?
12. What is the difference between a parameter and a hyperparameter?
13. Why is the test set not used for model selection?
14. What is automatic differentiation?
15. Why is autograd useful for PDEs?
16. What is a neural surrogate?
17. Why can a neural network fit data but still violate physics?
18. When might classical ML be preferable to deep learning?


# 68. Final challenge — Implement a tiny neural network manually

Without using `nn.Linear`, implement a single neuron using:

- a weight tensor,
- a bias tensor,
- matrix multiplication,
- a loss,
- autograd,
- an optimizer.

Then compare its result with `nn.Linear`.

This exercise makes the abstraction used by PyTorch transparent.


In [ ]:
torch.manual_seed(1)

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0]
])

y = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [9.0]
])

optimizer = torch.optim.SGD([w, b], lr=0.05)

for epoch in range(500):
    prediction = x * w + b
    loss = torch.mean((prediction - y)**2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Learned weight:", w.item())
print("Learned bias:", b.item())
print("Final loss:", loss.item())


# 69. Key takeaways

PyTorch is best understood as a framework for **tensor computation with automatic differentiation**.

The central workflow is:

**data → tensors → model → forward pass → loss → backward pass → optimizer → validation**

For computational materials, this expands to:

**physics → mathematical model → numerical/data representation → differentiable computation → learning/optimization → scientific validation**

The most important concepts to carry forward are:

- tensor shapes,
- vectorization,
- matrix operations,
- autograd,
- gradients,
- optimization,
- model architecture,
- data pipelines,
- validation,
- reproducibility,
- physical constraints.


# 70. Suggested next steps

After completing this notebook, proceed to:

1. **PyTorch deeper practice**
   - custom training loops
   - learning-rate schedulers
   - checkpointing
   - mixed precision

2. **Computer vision**
   - CNNs
   - transfer learning
   - segmentation
   - microscopy

3. **Graph neural networks**
   - atomic graphs
   - message passing
   - crystal representations

4. **Scientific machine learning**
   - PINNs
   - neural operators
   - differentiable PDE solvers

5. **Materials informatics**
   - pymatgen
   - matminer
   - ASE
   - materials datasets

6. **Atomistic ML**
   - graph/equivariant neural networks
   - interatomic potentials
   - property prediction

The goal is not merely to train a neural network. The goal is to use differentiable computation as a tool for solving meaningful materials-science problems.
